# Combining geographic layers

<style>
blockquote:has(.notebook-admonition-title) {
  --notebook-admonition-color: var(--color-admonition-title--note, #087fc7);
  --notebook-admonition-title-background:
    var(--color-admonition-title-background--note, rgba(8, 127, 199, 0.18));
  background: var(--color-admonition-background, transparent);
  border: 0;
  border-left: 0.2rem solid var(--notebook-admonition-color);
  border-radius: 0.2rem;
  box-shadow: 0 0.2rem 0.5rem rgba(0, 0, 0, 0.05), 0 0 0.0625rem rgba(0, 0, 0, 0.1);
  font-size: var(--admonition-font-size, 0.8125rem);
  margin: 1rem auto;
  overflow: hidden;
  padding: 0 0.5rem 0.5rem;
}
blockquote p:has(> .notebook-admonition-title) {
  background: var(--notebook-admonition-title-background);
  font-size: var(--admonition-title-font-size, 0.8125rem);
  font-weight: 500;
  line-height: 1.3;
  margin: 0 -0.5rem 0.5rem;
  padding: 0.4rem 0.5rem 0.4rem 2rem;
  position: relative;
}
blockquote p:has(> .notebook-admonition-title)::before {
  color: var(--notebook-admonition-color);
  content: "✎";
  left: 0.65rem;
  position: absolute;
}
.notebook-admonition-title {
  font-weight: inherit;
}
table:not(.dataframe) {
  border: 1px solid var(--docs-hairline, rgba(128, 128, 128, 0.35));
  border-collapse: collapse;
}
table:not(.dataframe) th,
table:not(.dataframe) td {
  border: 1px solid var(--docs-hairline, rgba(128, 128, 128, 0.35));
}
</style>

<div style="text-align: center;"><a class="sd-sphinx-override sd-btn sd-text-wrap sd-btn-primary reference external" href="https://github.com/mggg/gerrytools/tree/main/user_guide/_static/data">Browse tutorial data</a></div>

The geographic format guide shows each layer separately. This workflow combines a continuous
fill with district boundaries, district labels, and a focused extent. The result uses one
GeoDataFrame throughout, so layers align without manual geometry joins or coordinate changes.


In [ ]:
from pathlib import Path

import geopandas as gpd
import pandas as pd

from gerrytools.plotting import GeoPlot

precincts = gpd.read_file(Path("data/ga_2016_precincts.gpkg"))
precincts[["BVAP", "VAP"]] = precincts[["BVAP", "VAP"]].apply(pd.to_numeric)
precincts["BVAP_SHARE"] = precincts["BVAP"].div(precincts["VAP"])

The source rows are precincts. `BVAP_SHARE` is therefore calculated at precinct resolution,
while the existing `CD` column identifies the congressional district containing each precinct.
Those two columns support different layers on the same geometry.

## Compose the map

The choropleth is added first and supplies the continuous fill. The plan layer is transparent
inside each district, so only its dissolved boundaries and labels appear above that fill.
`default_outline=False` avoids drawing a third, redundant outline around every source unit.


In [ ]:
metro_mask = precincts["CTYNAME"].isin(["Clayton", "Cobb", "DeKalb", "Fulton", "Gwinnett"])

plot = GeoPlot(precincts, default_outline=False)
plot.add_choropleth_layer("BVAP_SHARE", colormap="Greys", vmin=0, vmax=1, show_colorbar=True)
plot.add_districting_plan_layer(
    "CD",
    dissolve=True,
    facealpha=0.5,
    show_labels=True,
    label_style="badge",
)
plot.focus_axes(geometry_mask=metro_mask)
plot.show()

`focus_axes()` changes only the visible extent. The plot still contains every precinct and
district, but the axes are fit to the five-county metro mask. The same statewide layers can
therefore support a statewide map and a regional detail without creating a second, filtered
GeoDataFrame.

Layer calls are stored by the builder, so the map can be revised after it renders. Adding a
highlight or changing the focus marks the builder for rebuilding on the next `.ax`, `show()`,
or `save()` call. The shared-controls guide covers those one-off adjustments.

## Related

- [Geographic plot formats](geo.ipynb)
- [Shared geographic controls](options.ipynb)
- [Matplotlib composition](../composition.ipynb)
